# banao-tech MuseTalk on a free Kaggle GPU — timing test

Kaggle (like Colab) now runs **Python 3.12**, but banao's working stack (torch 2.0.1, mmcv 2.0.1, transformers 4.39.2) is a **Python 3.10** stack — that mismatch is what caused the `is_offline_mode` import error. Fix: we build a **Python 3.10 virtualenv** (via `uv`, which downloads a standalone 3.10) and run banao's exact recipe inside it, on the GPU.

## Kaggle setup (right-hand panel, do first)
1. **Settings -> Accelerator -> GPU T4 x2** (or P100).
2. **Settings -> Internet -> On** (needs a phone-verified account).
3. **Add Input -> Upload -> New Dataset**: add your portrait (`luna.jpg`) + voice (`luna.wav`). They appear under `/kaggle/input/...` and cell 5 finds them.
4. Run cells top to bottom. The venv build (cell 3) takes several minutes.

In [ ]:
# 1. GPU + internet
!nvidia-smi
import urllib.request
try:
    urllib.request.urlopen('https://huggingface.co', timeout=10); print('Internet: ON')
except Exception as e:
    print('Internet: OFF ->', e, '\nTurn it on: Settings > Internet > On')

In [ ]:
# 2. System libs + clone the banao Space
!apt-get -y -qq install ffmpeg libgl1 libglib2.0-0 libsm6 libxext6 cmake build-essential git-lfs 2>/dev/null || true
%cd /kaggle/working
!rm -rf musetalk-avatar
!git clone https://huggingface.co/spaces/banao-tech/musetalk-avatar
%cd /kaggle/working/musetalk-avatar

In [ ]:
# 3. Build a PYTHON 3.10 venv with uv and install banao's stack (GPU) into it.
#    This is the fix: the base kernel is 3.12, but the venv is 3.10 where the wheels exist.
%cd /kaggle/working/musetalk-avatar
!pip install -q uv
!uv venv --python 3.10 /kaggle/working/py310
PY = "/kaggle/working/py310/bin/python"

!uv pip install --python {PY} numpy==1.26.4 torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!uv pip install --python {PY} "setuptools==68.2.2" wheel cython poetry-core
!uv pip install --python {PY} --no-build-isolation chumpy==0.70
!uv pip install --python {PY} mmcv==2.0.1 --find-links https://download.openmmlab.com/mmcv/dist/cu118/torch2.0.0/index.html
!grep -vE '^-f |^torch==|^torchvision==|^torchaudio==|^mmcv==|^numpy==' requirements.txt > reqs_gpu.txt
!uv pip install --python {PY} -r reqs_gpu.txt

print("\n===== verify inside the 3.10 venv =====")
!{PY} -c "import sys,torch,mmcv,mmdet,mmpose,transformers,huggingface_hub as h; print('python', sys.version.split()[0]); print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('mmpose', mmpose.__version__); print('transformers', transformers.__version__, 'hub', h.__version__)"

In [ ]:
# 4. Download weights (use the venv's python + huggingface-cli via PATH)
%cd /kaggle/working/musetalk-avatar
!MODEL_DIR=/kaggle/working/musetalk-avatar/models PATH=/kaggle/working/py310/bin:$PATH bash download_weights.sh

In [ ]:
# 5. Find your uploaded portrait + audio from the attached dataset (/kaggle/input)
import glob, os, shutil
%cd /kaggle/working/musetalk-avatar
imgs = sum([glob.glob(f"/kaggle/input/**/*.{e}", recursive=True) for e in ("jpg","jpeg","png")], [])
wavs = sum([glob.glob(f"/kaggle/input/**/*.{e}", recursive=True) for e in ("wav","mp3","m4a")], [])
assert imgs and wavs, "Attach a dataset with one image AND one audio (Add Input > Upload)."
os.makedirs("data/custom", exist_ok=True)
img, wav = os.path.basename(imgs[0]), os.path.basename(wavs[0])
shutil.copy(imgs[0], f"data/custom/{img}"); shutil.copy(wavs[0], f"data/custom/{wav}")
print("portrait:", imgs[0], "\naudio   :", wavs[0])

In [ ]:
# 6. Run banao's EXACT inference with the 3.10 venv python — timing + error capture.
import os, time, glob, base64, subprocess, soundfile as sf
from IPython.display import HTML
%cd /kaggle/working/musetalk-avatar

PY = "/kaggle/working/py310/bin/python"
MODEL_DIR = "/kaggle/working/musetalk-avatar/models"
os.makedirs("configs/inference", exist_ok=True)
yaml_text = f'''space_avatar:
  video_path: "data/custom/{img}"
  audio_path: "data/custom/{wav}"
  bbox_shift: 0
'''
open("configs/inference/kaggle_test.yaml", "w").write(yaml_text)
print(yaml_text)

info = sf.info(f"data/custom/{wav}"); audio_sec = info.frames / info.samplerate
cmd = [
    PY, "-m", "scripts.inference",
    "--inference_config", "configs/inference/kaggle_test.yaml",
    "--result_dir", "results/kaggle",
    "--unet_model_path", f"{MODEL_DIR}/musetalkV15/unet.pth",
    "--unet_config", f"{MODEL_DIR}/musetalkV15/musetalk.json",
    "--version", "v15", "--fps", "15", "--batch_size", "4", "--bbox_shift", "0",
    "--parsing_mode", "jaw", "--left_cheek_width", "90", "--right_cheek_width", "90",
]
env = os.environ.copy(); env["MODEL_DIR"] = MODEL_DIR
t0 = time.time(); r = subprocess.run(cmd, env=env, capture_output=True, text=True); dt = time.time() - t0

print(f"\n===== TIMING on this Kaggle GPU =====")
print(f"exit code        : {r.returncode}")
print(f"clip length      : {audio_sec:5.1f} s of audio")
print(f"generation time  : {dt:5.1f} s")
if audio_sec:
    print(f"realtime factor  : {dt/audio_sec:5.2f}x  (only meaningful if exit code is 0)")

vids = sorted(glob.glob("results/**/*.mp4", recursive=True), key=os.path.getmtime)
if r.returncode == 0 and vids:
    print("Showing:", vids[-1])
    b64 = base64.b64encode(open(vids[-1], 'rb').read()).decode()
    display(HTML(f'<video width=480 controls autoplay loop src="data:video/mp4;base64,{b64}"></video>'))
else:
    print("\n===== INFERENCE FAILED — last 4000 chars =====")
    print(((r.stdout or "") + "\n---- STDERR ----\n" + (r.stderr or ""))[-4000:])

### Read the result
- exit code 0 + a **realtime factor** = your real free-GPU speed number.
- If it still fails, paste the log. A 'no face detected' / empty-bbox error = the painterly portrait is the problem (try a photoreal face). A traceback = a setup fix.
- If even the 3.10 venv is too finicky, the guaranteed path is duplicating banao's Space onto a T4 GPU (its Docker is Python 3.10 by construction) — a few cents for the test.